In [0]:
%pip install psplib

Import Required Libraries

In [0]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from psplib import parse

Define Dataset Location

In [0]:
dataset_folder = "/Workspace/Users/aya.tlijani@esprim.tn/project-RCPSP/j30"

if os.path.exists(dataset_folder):
    print("Dataset folder found.")
else:
    print("Dataset folder not found.")

Count Project Instances

In [0]:
dataset_path = Path(dataset_folder)

sm_files = sorted(dataset_path.glob("*.sm"))

print(f"Total project instances: {len(sm_files)}")

Preview Dataset Files

In [0]:
preview = pd.DataFrame({
    "Project Instance": [file.name for file in sm_files]
})

display(preview.head(10))

Parse All Project Instances

In [0]:
projects = []
failed_files = []

for file in sm_files:
    try:
        project = parse(file)
        projects.append(project)
    except Exception as e:
        failed_files.append({
            "File": file.name,
            "Error": str(e)
        })

print(f"Successfully parsed: {len(projects)}")
print(f"Failed to parse: {len(failed_files)}")

if failed_files:
    display(pd.DataFrame(failed_files))

Inspect the Structure of a ProjectInstance

In [0]:
example_file = sm_files[0]
example_project = projects[0]

print(f"Loaded file: {example_file.name}")
print(f"Number of projects: {example_project.num_projects}")
print(f"Number of activities: {example_project.num_activities}")
print(f"Number of resources: {example_project.num_resources}")
print(f"Number of skills: {example_project.num_skills}")

print(f"""
Project ({example_file.stem})
│
├── Activities ({example_project.num_activities})
│
└── Resources ({example_project.num_resources})
""")

In [0]:
summary_rows = []

for file, project in zip(sm_files, projects):
    summary_rows.append({
        "Instance": file.stem,
        "Number of Activities": project.num_activities,
        "Number of Resources": project.num_resources,
    })

summary_df = pd.DataFrame(summary_rows)
display(summary_df.head(10))

In [0]:
project = projects[0]
n_resources = project.num_resources

activity_rows = []
for activity_id, activity in enumerate(project.activities):
    mode = activity.modes[0]  # single-mode RCPSP: only one mode per activity
    row = {"Activity ID": activity_id, "Duration": mode.duration}
    for r in range(n_resources):
        row[f"Resource {r + 1}"] = mode.demands[r]
    row["Successors"] = ",".join(map(str, activity.successors)) if activity.successors else ""
    activity_rows.append(row)

activity_df = pd.DataFrame(activity_rows)
display(activity_df)

In [0]:
all_durations = []
demand_by_resource = {}

for project in projects:
    for activity in project.activities:
        mode = activity.modes[0]
        all_durations.append(mode.duration)
        for r, demand in enumerate(mode.demands):
            demand_by_resource.setdefault(r, []).append(demand)

durations = np.array(all_durations)

print(f"Total activities analyzed: {len(durations)}")
print(f"Average duration: {durations.mean():.2f}")
print(f"Minimum duration: {durations.min()}")
print(f"Maximum duration: {durations.max()}")
print(f"Median duration:  {np.median(durations):.2f}")

In [0]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(durations, bins=range(0, int(durations.max()) + 2), edgecolor="black")
ax.set_xlabel("Duration")
ax.set_ylabel("Number of Activities")
ax.set_title("Distribution of Activity Durations (All Projects)")
plt.show()

In [0]:
resource_avg_demand = {
    f"Resource {r + 1}": np.mean(values)
    for r, values in sorted(demand_by_resource.items())
}

for name, avg in resource_avg_demand.items():
    print(f"{name} average demand = {avg:.2f}")

fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(list(resource_avg_demand.keys()), list(resource_avg_demand.values()))
ax.set_ylabel("Average Demand")
ax.set_title("Average Resource Demand per Activity (All Projects)")
plt.show()

In [0]:
resource_rows = []
for i, resource in enumerate(project.resources):
    resource_rows.append({
        "Resource": f"R{i + 1}",
        "Capacity": resource.capacity,
        "Renewable": resource.renewable,
    })

resource_df = pd.DataFrame(resource_rows)
display(resource_df)

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(resource_df["Resource"], resource_df["Capacity"])
ax.set_ylabel("Capacity")
ax.set_title(f"Resource Capacities — {sm_files[0].stem}")
plt.show()

In [0]:
n_activities = project.num_activities
n_dependencies = sum(len(activity.successors) for activity in project.activities)
avg_successors = n_dependencies / n_activities

print(f"Activities: {n_activities}")
print(f"Dependencies: {n_dependencies}")
print(f"Average connections: {avg_successors:.2f}")

In [0]:
%pip install networkx
import networkx as nx

G = nx.DiGraph()
for activity_id, activity in enumerate(project.activities):
    G.add_node(activity_id)
    for successor in activity.successors:
        G.add_edge(activity_id, successor)

fig, ax = plt.subplots(figsize=(10, 7))
pos = nx.spring_layout(G, seed=42, k=0.6)
nx.draw(
    G, pos, ax=ax, with_labels=True, node_color="#8ecae6",
    node_size=500, arrowsize=12, font_size=8,
)
ax.set_title(f"Scheduling Graph — {sm_files[0].stem}")
plt.show()

In [0]:
print("""
                 PROJECT
                    |
        -------------------------
        |                       |
    RESOURCES              ACTIVITIES
        |                       |
        |                       |
   Capacity                Activity ID
   Renewable               Duration
                            Resource Demand
                            Successors
""")

print(f"Dataset summary: {len(projects)} projects parsed from '{dataset_folder}'")
print(f"Each project has {project.num_activities} activities and {project.num_resources} renewable resources (typical for J30).")